# Content-Based Recommendation System & Profile Evaluation
### Smart Trip Planning System
This notebook implements and benchmarks the machine learning engine:
1. **TF-IDF Vectorization** (`stop_words='english'`, `ngram_range=(1, 2)`)
2. **Cosine Similarity Matrix** between query vector and attraction feature vectors
3. **Multi-Factor Weighted Scoring Equation** incorporating:
   - Interest Match (Content similarity + category direct match)
   - Normalized Rating
   - Budget Suitability
   - Popularity Index
   - Geodesic Distance from Destination Center
4. **Joblib Serialization** to save `recommendation_model.pkl`
5. **Real Testing Across 5 Distinct User Profiles** with calculated Precision@3 benchmark


In [ ]:
import os
import re
import math
import joblib
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = '../data'
MODELS_DIR = '../models'
attractions = pd.read_csv(os.path.join(DATA_DIR, 'attractions.csv'))
print(f'Loaded attractions: {attractions.shape}')


## 1. Feature Preprocessing & TF-IDF Vectorization


In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

attractions['combined_features'] = attractions.apply(
    lambda r: clean_text(f"{r['category']} {r['category']} {r['tags']} {r['tags']} {r['description']} {r['indoor_outdoor']}"),
    axis=1
)

# Fit TF-IDF Vectorizer with unigrams and bigrams
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(attractions['combined_features'])

print(f'TF-IDF Feature Space Dimensions: {tfidf_matrix.shape}')
print('Sample learned vocabulary tokens:', list(vectorizer.vocabulary_.keys())[:15])


## 2. Train & Save Model Artifacts with Joblib


In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
model_data = {
    'vectorizer': vectorizer,
    'tfidf_matrix': tfidf_matrix,
    'attractions_df': attractions
}
joblib_path = os.path.join(MODELS_DIR, 'recommendation_model.pkl')
joblib.dump(model_data, joblib_path)
print(f'Model artifact successfully saved to: {joblib_path}')

# Verify loading
loaded_data = joblib.load(joblib_path)
assert loaded_data['tfidf_matrix'].shape == tfidf_matrix.shape
print('Verified: Saved model loads cleanly from disk.')


## 3. Weighted Recommendation Scoring Function


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2.0)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return round(R * c, 2)

def recommend_attractions(destination, interests, budget=15000, days=3, travelers=2, top_k=5):
    df = attractions[attractions['destination'].str.lower() == destination.lower()].copy().reset_index(drop=True)
    if df.empty: return pd.DataFrame()
    
    dest_center_lat = df['latitude'].mean()
    dest_center_lon = df['longitude'].mean()
    
    # User query vector
    query_text = ' '.join(interests) + ' ' + ' '.join([f'{i} {i}' for i in interests])
    query_vec = vectorizer.transform([clean_text(query_text)])
    cand_mat = vectorizer.transform(df['combined_features'])
    sim_scores = cosine_similarity(query_vec, cand_mat).flatten()
    
    est_attr_budget = max(400.0, (budget * 0.20) / max(1, days * 3 * travelers))
    user_ints_lower = [i.lower() for i in interests]
    
    results = []
    for idx, row in df.iterrows():
        # 1. Interest Match (TF-IDF Similarity + Category Match)
        c_sim = float(sim_scores[idx])
        cat_match = 1.0 if row['category'].lower() in user_ints_lower else 0.2
        interest_score = 0.6 * c_sim + 0.4 * cat_match
        
        # 2. Rating Factor
        r_score = max(0.0, min(1.0, (row['rating'] - 3.5) / 1.5))
        
        # 3. Budget Suitability
        price = float(row['price'])
        b_score = 1.0 if price == 0 else max(0.2, 1.0 - min(1.0, price / (est_attr_budget * 2)))
        
        # 4. Popularity
        pop_score = max(0.0, min(1.0, (row['popularity'] - 50) / 50))
        
        # 5. Distance Factor
        dist_km = haversine(dest_center_lat, dest_center_lon, row['latitude'], row['longitude'])
        dist_score = max(0.2, 1.0 - min(1.0, dist_km / 30.0))
        
        # Final Weighted Score
        final_score = (
            0.40 * interest_score +
            0.20 * r_score +
            0.15 * b_score +
            0.15 * pop_score +
            0.10 * dist_score
        )
        
        results.append({
            'Name': row['name'],
            'Category': row['category'],
            'Rating': row['rating'],
            'Price': row['price'],
            'Distance_km': dist_km,
            'Interest_Match_%': round(interest_score * 100, 1),
            'Score_%': round(final_score * 100, 1)
        })
    
    res_df = pd.DataFrame(results)
    return res_df.sort_values(by='Score_%', ascending=False).reset_index(drop=True).head(top_k)


## 4. Model Evaluation with 5 Distinct User Profiles


In [ ]:
test_profiles = [
    {'Profile': 'Profile 1: Adventure & Nature Couple', 'Destination': 'Ooty', 'Interests': ['Nature', 'Adventure', 'Photography'], 'Budget': 15000, 'Days': 3, 'Travelers': 2},
    {'Profile': 'Profile 2: Beach & Culture Group', 'Destination': 'Goa', 'Interests': ['Beach', 'Culture', 'Historical'], 'Budget': 25000, 'Days': 4, 'Travelers': 3},
    {'Profile': 'Profile 3: Royal Heritage Solo Traveler', 'Destination': 'Jaipur', 'Interests': ['Historical', 'Museum', 'Photography'], 'Budget': 18000, 'Days': 3, 'Travelers': 1},
    {'Profile': 'Profile 4: Spiritual Pilgrim', 'Destination': 'Varanasi', 'Interests': ['Temple', 'Culture'], 'Budget': 10000, 'Days': 2, 'Travelers': 2},
    {'Profile': 'Profile 5: Snow & Trekking Explorer', 'Destination': 'Manali', 'Interests': ['Adventure', 'Nature'], 'Budget': 20000, 'Days': 4, 'Travelers': 2}
]

summary_rows = []
for p in test_profiles:
    recs = recommend_attractions(p['Destination'], p['Interests'], p['Budget'], p['Days'], p['Travelers'], top_k=3)
    user_ints_lower = set([i.lower() for i in p['Interests']])
    rel_count = sum(recs['Category'].str.lower().isin(user_ints_lower))
    p_at_3 = rel_count / len(recs) if len(recs) > 0 else 0.0
    
    top_pick = recs.iloc[0]['Name'] if len(recs) > 0 else 'N/A'
    top_score = recs.iloc[0]['Score_%'] if len(recs) > 0 else 0.0
    
    summary_rows.append({
        'Profile': p['Profile'],
        'Destination': p['Destination'],
        'Interests': ', '.join(p['Interests']),
        'Top Pick': top_pick,
        'Top Score (%)': top_score,
        'Precision@3': round(p_at_3, 2)
    })

summary_benchmark = pd.DataFrame(summary_rows)
summary_benchmark


## 5. Visualizing Benchmark Comparison Across Profiles


In [ ]:
plt.figure(figsize=(10, 5))
palette = sns.color_palette('crest', len(summary_benchmark))
bars = plt.bar(summary_benchmark['Destination'], summary_benchmark['Top Score (%)'], color=palette, edgecolor='black')
plt.title('Top Recommendation Score by Destination Profile', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Destination & User Context', fontsize=11)
plt.ylabel('Top Attraction Score (%)', fontsize=11)
plt.ylim(50, 100)
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, y + 1.0, f'{y:.1f}%', ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()
